## 解説

説明のために
3x3のmeshを考えそのセルが値を持つとします。
3x3行例$w$のindexを以下で定義します。

![w定義](image_keep/mesh_w_index.png "w定義" )

並行ビーム系tomograpy像は様々にwを回転させて測定した断面画像です。
例えば、下の図では$y_1,y_2,...y_8$の像が撮影されます。

![w回転](image_keep/mesh_rotation.png "w回転")

このようなトモグラフ像から逆に$w$の値を求めることを画像の復元、もしくは再構成といいます。

物質が回転していない（$\theta=0$）の場合を
$\vec{y}_{\theta=0}= (y_2,y_3,y_4)$と
$w = (w_1, w_2, w_3, \cdots, w_{9})^T$に対して
例えばaの配置の場合は
$$
X_{\theta=0} = \left( \begin{matrix}
0,0,0,& 0,0,0,& 0,0,0  \\
1,1,1,& 0,0,0,& 0,0,0  \\
0,0,0,& 1,1,1,& 0,0,0  \\
0,0,0,& 0,0,0,& 1,1,1  \\
0,0,0,& 0,0,0,& 0,0,0  
 \end{matrix} \right)
$$
を用いて$\vec{y}_{\theta=0} = X_{\theta=0} \vec{w}$という関係になります。
この変換行列は$w$の値に依らず回転角$\theta$に依ります。
一般的の回転角の場合の変換行列$X$は予め求めておくことが可能であり、
全ての回転角を合わせても$\vec{y} = X \vec{w}$という関係になります。


### Xの求め方

scikit-learnの例では以下の寄与を計算しています．
bの配置の場合に
下図の黒四角mesh $w_6$の中心座標の寄与を中心の両隣のbinに$w_6$の寄与を分配すると考えて変換行列$X$を計算する。
#### 例、中心座標x＝3.2の場合
![Xの寄与](image_keep/X_contribution.PNG "Xの寄与")
* floor(x)=2番目のbinに (1-frac(x))の寄与、つまり、$w_6$=0.8$w_6$
* floor(x)+1=3番目のbinにfrac(x)の寄与、つまり、$w_6$=0.2$w_6$

の寄与をするとしている。

### 解き方

先程書いたとおり、
tomography像撮影は
$$
 \vec{y} = X \vec{w}
$$
という変換です。
つまり、
$\vec{y}$のサイズ（方程式の数）を$P$、$\vec{w}$のサイズ（未知数）をN、
行列$X$のサイズを$(P,N)$と置き、コードでもこの変数を用いると。

$$
y_i = \sum_{j=1}^{N} X_{i,j}  w_j
$$

という線形方程式に対応します。

$w$を求めるには、以下の解き方が考えられます。

1. $N=P$の場合に

$$
\vec{w} = X^{-1} \vec{y}
$$

とできます。

（**デーインスタンス数？データインスタンスサイズ？**）

1. 観測データにはノイズが乗るため観測データ数をなるべく増やします。多くの場合に $P>N$ の関係になります。
この関係の場合には$\vec{y}-X \vec{w}$の自乗誤差を最小化する問題

$$
  f = \mbox{argmin}_{w} [|\!| \vec{y} - X \vec{w} |\!|^2 ]
$$

を解きます。更に，最もなめらかな像（$w$）を得るために多くの場合にエントロピー最大化の付加条件を加えます。

2. 上の表式は線形回帰の最適化関数と同じです。
$\vec{w}$の解空間がスパースである（０が多い）場合にL1罰則項を追加すると一意に近似解を得ることができます。

$$
  f_{L1} = \mbox{argmin}_{w} [|\!| \vec{y} - X \vec{w} |\!|^2  + \alpha |\!|\vec{w}|\!|_1]
$$

なお、$P<N$でもL1罰則項を加えると一意に解が得られます。

ここでは２番めの解き方を実行します。
